In [0]:
%pip install tqdm
%pip install databricks-sdk[openai] --upgrade

dbutils.library.restartPython()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/58.7 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.7/58.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/78.6 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 2.6 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/43.5 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.4 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/42.0 kB ? eta -:--:--
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/323.4 kB ? eta -:--:--
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/

In [0]:
spark.conf.set('spark.sql.session.timeZone', 'Australia/Sydney')
import os, time
os.environ["TZ"] = "Australia/Sydney"
time.tzset()

In [0]:
# The standard task instruction
RMET_INSTRUCTION = """For this set of eyes, choose which word best describes what the person in the picture is thinking or feeling. You may feel that more than one word is applicable but please choose just one word, the word which you consider to be most suitable. Before making your choice, make sure that you have read all 4 words. If you really don't know what a word means you can consult the definitions provided."""

# Each question = 1 image + 4 word options + correct answer (for scoring)
QUESTIONS = [
    {"id": "rmet-1",  "image": "eyes_1.png",  "options": ["playful", "comforting", "irritated", "bored"],            "correct": "playful"},
    {"id": "rmet-2",  "image": "eyes_2.png",  "options": ["terrified", "upset", "arrogant", "annoyed"],              "correct": "upset"},
    {"id": "rmet-3",  "image": "eyes_3.png",  "options": ["joking", "flustered", "desire", "convinced"],             "correct": "desire"},
    {"id": "rmet-4",  "image": "eyes_4.png",  "options": ["joking", "insisting", "amused", "relaxed"],               "correct": "insisting"},
    {"id": "rmet-5",  "image": "eyes_5.png",  "options": ["irritated", "sarcastic", "worried", "friendly"],          "correct": "worried"},
    {"id": "rmet-6",  "image": "eyes_6.png",  "options": ["aghast", "fantasizing", "impatient", "alarmed"],          "correct": "fantasizing"},
    {"id": "rmet-7",  "image": "eyes_7.png",  "options": ["apologetic", "friendly", "uneasy", "dispirited"],         "correct": "uneasy"},
    {"id": "rmet-8",  "image": "eyes_8.png",  "options": ["despondent", "relieved", "shy", "excited"],               "correct": "despondent"},
    {"id": "rmet-9",  "image": "eyes_9.png",  "options": ["annoyed", "hostile", "horrified", "preoccupied"],         "correct": "preoccupied"},
    {"id": "rmet-10", "image": "eyes_10.png", "options": ["cautious", "insisting", "bored", "aghast"],               "correct": "cautious"},
    {"id": "rmet-11", "image": "eyes_11.png", "options": ["terrified", "amused", "regretful", "flirtatious"],        "correct": "regretful"},
    {"id": "rmet-12", "image": "eyes_12.png", "options": ["indifferent", "embarrassed", "sceptical", "dispirited"],  "correct": "sceptical"},
    {"id": "rmet-13", "image": "eyes_13.png", "options": ["decisive", "anticipating", "threatening", "shy"],         "correct": "anticipating"},
    {"id": "rmet-14", "image": "eyes_14.png", "options": ["irritated", "disappointed", "depressed", "accusing"],     "correct": "accusing"},
    {"id": "rmet-15", "image": "eyes_15.png", "options": ["contemplative", "flustered", "encouraging", "amused"],    "correct": "contemplative"},
    {"id": "rmet-16", "image": "eyes_16.png", "options": ["irritated", "thoughtful", "encouraging", "sympathetic"],  "correct": "thoughtful"},
    {"id": "rmet-17", "image": "eyes_17.png", "options": ["doubtful", "affectionate", "playful", "aghast"],          "correct": "doubtful"},
    {"id": "rmet-18", "image": "eyes_18.png", "options": ["decisive", "amused", "aghast", "bored"],                  "correct": "decisive"},
    {"id": "rmet-19", "image": "eyes_19.png", "options": ["arrogant", "grateful", "sarcastic", "tentative"],         "correct": "tentative"},
    {"id": "rmet-20", "image": "eyes_20.png", "options": ["dominant", "friendly", "guilty", "horrified"],            "correct": "friendly"},
    {"id": "rmet-21", "image": "eyes_21.png", "options": ["embarrassed", "fantasizing", "confused", "panicked"],     "correct": "fantasizing"},
    {"id": "rmet-22", "image": "eyes_22.png", "options": ["preoccupied", "grateful", "insisting", "imploring"],      "correct": "preoccupied"},
    {"id": "rmet-23", "image": "eyes_23.png", "options": ["contented", "apologetic", "defiant", "curious"],          "correct": "defiant"},
    {"id": "rmet-24", "image": "eyes_24.png", "options": ["pensive", "irritated", "excited", "hostile"],             "correct": "pensive"},
    {"id": "rmet-25", "image": "eyes_25.png", "options": ["panicked", "incredulous", "despondent", "interested"],    "correct": "interested"},
    {"id": "rmet-26", "image": "eyes_26.png", "options": ["alarmed", "shy", "hostile", "anxious"],                   "correct": "hostile"},
    {"id": "rmet-27", "image": "eyes_27.png", "options": ["joking", "cautious", "arrogant", "reassuring"],           "correct": "cautious"},
    {"id": "rmet-28", "image": "eyes_28.png", "options": ["interested", "joking", "affectionate", "contented"],      "correct": "interested"},
    {"id": "rmet-29", "image": "eyes_29.png", "options": ["impatient", "aghast", "irritated", "reflective"],         "correct": "reflective"},
    {"id": "rmet-30", "image": "eyes_30.png", "options": ["grateful", "flirtatious", "hostile", "disappointed"],     "correct": "flirtatious"},
    {"id": "rmet-31", "image": "eyes_31.png", "options": ["ashamed", "confident", "joking", "dispirited"],           "correct": "confident"},
    {"id": "rmet-32", "image": "eyes_32.png", "options": ["serious", "ashamed", "bewildered", "alarmed"],            "correct": "serious"},
    {"id": "rmet-33", "image": "eyes_33.png", "options": ["embarrassed", "guilty", "fantasizing", "concerned"],      "correct": "concerned"},
    {"id": "rmet-34", "image": "eyes_34.png", "options": ["aghast", "baffled", "distrustful", "terrified"],          "correct": "distrustful"},
    {"id": "rmet-35", "image": "eyes_35.png", "options": ["puzzled", "nervous", "insisting", "contemplative"],       "correct": "nervous"},
    {"id": "rmet-36", "image": "eyes_36.png", "options": ["ashamed", "nervous", "suspicious", "indecisive"],         "correct": "suspicious"},
]

IMAGE_BASE = "<path to RMET image folder>/"

# ── Full definitions dictionary (extend with all 36+ words) ──
WORD_DEFINITIONS = {
    "accusing":      "blaming. (e.g. The policeman was accusing the man of stealing a wallet.)",
    "affectionate":  "showing fondness towards someone. (e.g. Most mothers are affectionate to their babies by giving them lots of kisses and cuddles.)",
    "aghast":        "horrified, astonished, alarmed. (e.g. Jane was aghast when she discovered her house had been burgled.)",
    "alarmed":       "fearful, worried, filled with anxiety. (e.g. Claire was alarmed when she thought she was being followed home.)",
    "amused":        "finding something funny. (e.g. I was amused by a funny joke someone told me.)",
    "annoyed":       "irritated, displeased. (e.g. Jack was annoyed when he found out he had missed the last bus home.)",
    "anticipating":  "expecting. (e.g. At the start of the football match, the fans were anticipating a quick goal.)",
    "anxious":       "worried, tense, uneasy. (e.g. The student was feeling anxious before taking her final exams.)",
    "apologetic":    "feeling sorry. (e.g. The waiter was very apologetic when he spilt soup all over the customer.)",
    "arrogant":      "conceited, self-important, having a big opinion of oneself. (e.g. The arrogant man thought he knew more about politics than everyone else in the room.)",
    "ashamed":       "overcome with shame or guilt. (e.g. The boy felt ashamed when his mother discovered him stealing money from her purse.)",

    "assertive":     "confident, dominant, sure of oneself. (e.g. The assertive woman demanded that the shop give her a refund.)",
    "baffled":       "confused, puzzled, dumbfounded. (e.g. The detectives were completely baffled by the murder case.)",
    "bewildered":    "utterly confused, puzzled, dazed. (e.g. The child was bewildered when visiting the big city for the first time.)",
    "cautious":      "careful, wary. (e.g. Sarah was always a bit cautious when talking to someone she did not know.)",
    "comforting":    "consoling, compassionate. (e.g. The nurse was comforting the wounded soldier.)",
    "concerned":     "worried, troubled. (e.g. The doctor was concerned when his patient took a turn for the worse.)",
    "confident":     "self-assured, believing in oneself. (e.g. The tennis player was feeling very confident about winning his match.)",
    "confused":      "puzzled, perplexed. (e.g. Lizzie was so confused by the directions given to her, she got lost.)",
    "contemplative": "reflective, thoughtful, considering. (e.g. John was in a contemplative mood on the eve of his 60th birthday.)",
    "contented":     "satisfied. (e.g. After a nice walk and a good meal, David felt very contented.)",
    "convinced":     "certain, absolutely positive. (e.g. Richard was convinced he had come to the right decision.)",
    "curious":       "inquisitive, inquiring, prying. (e.g. Louise was curious about the strange shaped parcel.)",
    "deciding":      "making your mind up. (e.g. The man was deciding whom to vote for in the election.)",

    "decisive":      "already made your mind up. (e.g. Jane looked very decisive as she walked into the polling station.)",
    "defiant":       "insolent, bold, don't care what anyone else thinks. (e.g. The animal protester remained defiant even after being sent to prison.)",
    "depressed":     "miserable. (e.g. George was depressed when he didn't receive any birthday cards.)",
    "desire":        "passion, lust, longing for. (e.g. Kate had a strong desire for chocolate.)",
    "despondent":    "gloomy, despairing, without hope. (e.g. Gary was despondent when he did not get the job he wanted.)",
    "disappointed":  "displeased, disgruntled. (e.g. Manchester United fans were disappointed not to win the Championship.)",
    "dispirited":    "glum, miserable, low. (e.g. Adam was dispirited when he failed his exams.)",
    "distrustful":   "suspicious, doubtful, wary. (e.g. The old woman was distrustful of the stranger at her door.)",
    "dominant":      "commanding, bossy. (e.g. The sergeant major looked dominant as he inspected the new recruits.)",
    "doubtful":      "dubious, suspicious, not really believing. (e.g. Mary was doubtful that her son was telling the truth.)",
    "dubious":       "doubtful, suspicious. (e.g. Peter was dubious when offered a surprisingly cheap television in a pub.)",
    "eager":         "keen. (e.g. On Christmas morning, the children were eager to open their presents.)",
    "earnest":       "having a serious intention. (e.g. Harry was very earnest about his religious beliefs.)",

    "embarrassed":   "ashamed. (e.g. After forgetting a colleague's name, Jenny felt very embarrassed.)",
    "encouraging":   "hopeful, heartening, supporting. (e.g. All the parents were encouraging their children in the school sports day.)",
    "entertained":   "absorbed and amused or pleased by something. (e.g. I was very entertained by the magician.)",
    "enthusiastic":  "very eager, keen. (e.g. Susan felt very enthusiastic about her new fitness plan.)",
    "fantasizing":   "daydreaming. (e.g. Emma was fantasizing about being a film star.)",
    "fascinated":    "captivated, really interested. (e.g. At the seaside, the children were fascinated by the creatures in the rock pools.)",
    "fearful":       "terrified, worried. (e.g. In the dark streets, the women felt fearful.)",
    "flirtatious":   "brazen, saucy, teasing, playful. (e.g. Connie was accused of being flirtatious when she winked at a stranger at a party.)",
    "flustered":     "confused, nervous and upset. (e.g. Sarah felt a bit flustered when she realised how late she was for the meeting and that she had forgotten an important document.)",
    "friendly":      "sociable, amiable. (e.g. The friendly girl showed the tourists the way to the town centre.)",
    "grateful":      "thankful. (e.g. Kelly was very grateful for the kindness shown by the stranger.)",
    "guilty":        "feeling sorry for doing something wrong. (e.g. Charlie felt guilty about having an affair.)",
    "hateful":       "showing intense dislike. (e.g. The two sisters were hateful to each other and always fighting.)",

    "hopeful":       "optimistic. (e.g. Larry was hopeful that the post would bring good news.)",
    "horrified":     "terrified, appalled. (e.g. The man was horrified to discover that his new wife was already married.)",
    "hostile":       "unfriendly. (e.g. The two neighbours were hostile towards each other because of an argument about loud music.)",
    "impatient":     "restless, wanting something to happen soon. (e.g. Jane grew increasingly impatient as she waited for her friend who was already 20 minutes late.)",
    "imploring":     "begging, pleading. (e.g. Nicola looked imploring as she tried to persuade her dad to lend her the car.)",
    "incredulous":   "not believing. (e.g. Simon was incredulous when he heard that he had won the lottery.)",
    "indecisive":    "unsure, hesitant, unable to make your mind up. (e.g. Tammy was so indecisive that she couldn't even decide what to have for lunch.)",
    "indifferent":   "disinterested, unresponsive, don't care. (e.g. Terry was completely indifferent as to whether they went to the cinema or the pub.)",
    "insisting":     "demanding, persisting, maintaining. (e.g. After a work outing, Frank was insisting he paid the bill for everyone.)",
    "insulting":     "rude, offensive. (e.g. The football crowd was insulting the referee after he gave a penalty.)",
    "interested":    "inquiring, curious. (e.g. After seeing Jurassic Park, Hugh grew very interested in dinosaurs.)",
    "intrigued":     "very curious, very interested. (e.g. A mystery phone call intrigued Zoe.)",

    "irritated":     "exasperated, annoyed. (e.g. Frances was irritated by all the junk mail she received.)",
    "jealous":       "envious. (e.g. Tony was jealous of all the taller, better-looking boys in his class.)",
    "joking":        "being funny, playful. (e.g. Gary was always joking with his friends.)",
    "nervous":       "apprehensive, tense, worried. (e.g. Just before her job interview, Alice felt very nervous.)",
    "offended":      "insulted, wounded, having hurt feelings. (e.g. When someone made a joke about her weight, Martha felt very offended.)",
    "panicked":      "distraught, feeling of terror or anxiety. (e.g. On waking to find the house on fire, the whole family was panicked.)",
    "pensive":       "thinking about something slightly worrying. (e.g. Susie looked pensive on the way to meeting her boyfriend's parents for the first time.)",
    "perplexed":     "bewildered, puzzled, confused. (e.g. Frank was perplexed by the disappearance of his garden gnomes.)",
    "playful":       "full of high spirits and fun. (e.g. Neil was feeling playful at his birthday party.)",
    "preoccupied":   "absorbed, engrossed in one's own thoughts. (e.g. Worrying about her mother's illness made Debbie preoccupied at work.)",
    "puzzled":       "perplexed, bewildered, confused. (e.g. After doing the crossword for an hour, June was still puzzled by one clue.)",
    "reassuring":    "supporting, encouraging, giving someone confidence. (e.g. Andy tried to look reassuring as he told his wife that her new dress did suit her.)",

    "reflective":    "contemplative, thoughtful. (e.g. George was in a reflective mood as he thought about what he'd done with his life.)",
    "regretful":     "sorry. (e.g. Lee was always regretful that he had never travelled when he was younger.)",
    "relaxed":       "taking it easy, calm, carefree. (e.g. On holiday, Pam felt happy and relaxed.)",
    "relieved":      "freed from worry or anxiety. (e.g. At the restaurant, Ray was relieved to find that he had not forgotten his wallet.)",
    "resentful":     "bitter, hostile. (e.g. The businessman felt very resentful towards his younger colleague who had been promoted above him.)",
    "sarcastic":     "cynical, mocking, scornful. (e.g. The comedian made a sarcastic comment when someone came into the theatre late.)",
    "satisfied":     "content, fulfilled. (e.g. Steve felt very satisfied after he had got his new flat just how he wanted it.)",
    "sceptical":     "doubtful, suspicious, mistrusting. (e.g. Patrick looked sceptical as someone read out his horoscope to him.)",
    "serious":       "solemn, grave. (e.g. The bank manager looked serious as he refused Nigel an overdraft.)",
    "stern":         "severe, strict, firm. (e.g. The teacher looked very stern as he told the class off.)",
    "suspicious":    "disbelieving, suspecting, doubting. (e.g. After Sam had lost his wallet for the second time at work, he grew suspicious of one of his colleagues.)",
    "sympathetic":   "kind, compassionate. (e.g. The nurse looked sympathetic as she told the patient the bad news.)",

    "tentative":     "hesitant, uncertain, cautious. (e.g. Andrew felt a bit tentative as he went into the room full of strangers.)",
    "terrified":     "alarmed, fearful. (e.g. The boy was terrified when he thought he saw a ghost.)",
    "thoughtful":    "thinking about something. (e.g. Phil looked thoughtful as he sat waiting for the girlfriend he was about to finish with.)",
    "threatening":   "menacing, intimidating. (e.g. The large, drunken man was acting in a very threatening way.)",
    "uneasy":        "unsettled, apprehensive, troubled. (e.g. Karen felt slightly uneasy about accepting a lift from the man she had only met that day.)",
    "upset":         "agitated, worried, uneasy. (e.g. The man was very upset when his mother died.)",
    "worried":       "anxious, fretful, troubled. (e.g. When her cat went missing, the girl was very worried.)",
}


In [0]:
def validate_definitions():
    """Check all option words across all questions exist in WORD_DEFINITIONS."""
    used_words = set()
    for q in QUESTIONS:
        for w in q["options"]:
            used_words.add(w.lower())
        used_words.add(q["correct"].lower())

    missing = sorted(used_words - set(WORD_DEFINITIONS.keys()))
    if missing:
        print(f"⚠️  {len(missing)} words missing from WORD_DEFINITIONS:")
        for w in missing:
            print(f"   - {w}")
    else:
        print(f"✅ All {len(used_words)} unique words have definitions.")

    # Also report unused definitions (informational)
    unused = sorted(set(WORD_DEFINITIONS.keys()) - used_words)
    print(f"\nℹ️  {len(unused)} definitions in dictionary not used by current questions.")
    return missing

validate_definitions()

⚠️  3 words missing from WORD_DEFINITIONS:
   - bored
   - excited
   - shy

ℹ️  19 definitions in dictionary not used by current questions.


['bored', 'excited', 'shy']

In [0]:
import base64
import os

def encode_image_to_base64(image_path: str) -> str:
    """Read an image file and return a base64 data URL string."""
    with open(image_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    ext = os.path.splitext(image_path)[1].lower().lstrip(".")
    # Normalize mime type
    if ext in ("jpg", "jpeg"):
        mime = "jpeg"
    elif ext == "png":
        mime = "png"
    else:
        mime = ext  # gif, webp, etc.
    return f"data:image/{mime};base64,{encoded}"


def build_rmet_prompt(question: dict, include_definitions: bool = False) -> str:
    options_text = ", ".join(f'"{w}"' for w in question["options"])

    defs_block = ""
    if include_definitions:
        lines = []
        for w in question["options"]:
            d = WORD_DEFINITIONS.get(w.lower())
            if d:
                lines.append(f'- "{w}": {d}')
        if lines:
            defs_block = "\n\nDefinitions:\n" + "\n".join(lines)

    return f"""{RMET_INSTRUCTION}

The four word options are: {options_text}{defs_block}

Respond with ONLY a JSON object in this exact format:
{{"choice": "<one of the four words>"}}

Do not include any explanation or additional text."""

In [0]:
from abc import ABC, abstractmethod

class LLMClient(ABC):
    @abstractmethod
    def query(self, prompt: str, image_data_url: str = None, system: str = None) -> str:
        pass


# ---------- Azure OpenAI (GPT vision models) ----------
class AzureOpenAIClient(LLMClient):
    def __init__(self, api_key, endpoint, deployment, api_version="2024-12-01-preview"):
        from openai import AzureOpenAI
        self.client = AzureOpenAI(
            api_version=api_version,
            azure_endpoint=endpoint,
            api_key=api_key,
        )
        self.deployment = deployment

    def query(self, prompt: str, image_data_url: str = None, system: str = None) -> str:
        user_content = [{"type": "text", "text": prompt}]
        if image_data_url:
            user_content.append({
                "type": "image_url",
                "image_url": {"url": image_data_url}
            })
        resp = self.client.chat.completions.create(
            model=self.deployment,
            messages=[
                {"role": "system", "content": system or "You are completing a psychological task."},
                {"role": "user", "content": user_content}
            ]
        )
        return resp.choices[0].message.content


# ---------- Databricks Model Serving (vision-capable models) ----------
class DatabricksServingClient(LLMClient):
    def __init__(self, endpoint_name: str):
        from openai import OpenAI
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        token = ctx.apiToken().get()
        base_url = f"{ctx.apiUrl().get()}/serving-endpoints"
        self.client = OpenAI(api_key=token, base_url=base_url)
        self.endpoint_name = endpoint_name

    def query(self, prompt: str, image_data_url: str = None, system: str = None) -> str:
        user_content = [{"type": "text", "text": prompt}]
        if image_data_url:
            user_content.append({
                "type": "image_url",
                "image_url": {"url": image_data_url}
            })
        resp = self.client.chat.completions.create(
            model=self.endpoint_name,
            messages=[
                {"role": "system", "content": system or "You are completing a psychological task."},
                {"role": "user", "content": user_content}
            ]
        )
        return resp.choices[0].message.content


def get_client(provider: str, **kwargs) -> LLMClient:
    if provider == "azure_openai":
        return AzureOpenAIClient(
            api_key=kwargs["api_key"],
            endpoint=kwargs["endpoint"],
            deployment=kwargs["deployment"],
            api_version=kwargs.get("api_version", "2024-12-01-preview"),
        )
    elif provider == "databricks":
        return DatabricksServingClient(endpoint_name=kwargs["endpoint_name"])
    raise ValueError(f"Unknown provider: {provider}")

In [0]:
import re
import json

def parse_rmet_response(raw, options: list) -> dict:
    """Parse the model's chosen word, with fallbacks."""
    # Normalize list/dict responses to string
    if isinstance(raw, list):
        parts = []
        for item in raw:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(str(item.get("text", item.get("content", ""))))
            else:
                parts.append(str(item))
        raw = " ".join(parts)
    elif not isinstance(raw, str):
        raw = str(raw)

    result = {"choice": None, "raw": raw, "parse_error": None}
    options_lower = {w.lower(): w for w in options}

    # Try JSON first
    try:
        json_match = re.search(r'\{[^}]+\}', raw)
        if json_match:
            data = json.loads(json_match.group())
            choice = str(data.get("choice", "")).strip().lower()
            if choice in options_lower:
                result["choice"] = options_lower[choice]
                return result
    except (json.JSONDecodeError, ValueError, AttributeError):
        pass

    # Fallback: search for any option word appearing in the text
    raw_lower = raw.lower()
    for w_lower, w_orig in options_lower.items():
        if re.search(rf'\b{re.escape(w_lower)}\b', raw_lower):
            result["choice"] = w_orig
            return result

    result["parse_error"] = "Could not parse a valid option"
    return result

In [0]:
from datetime import datetime
import time
import os

AZURE_CONFIG = dict(
    api_key='###',
    endpoint="###",
    api_version="2024-12-01-preview",
)

# List model IDs per provider. Client is built automatically.
PROVIDERS = {
    "azure": [
        "gpt-5.5",
        "gpt-5.4",
        "gpt-5.2",
        "gpt-5.1",
        "gpt-5",
        "gpt-4o",
        "gpt-4.1",
    ],
    "databricks": [
        "databricks-claude-opus-4-8",
        "databricks-claude-opus-4-7",
        "databricks-claude-opus-4-6",
        "databricks-claude-opus-4-5",
        "databricks-claude-opus-4-1",
        "databricks-claude-sonnet-4-6",
        "databricks-claude-sonnet-4-5",
        "databricks-claude-sonnet-4",
        "databricks-claude-haiku-4-5",
        "databricks-gemma-3-12b",
        "databricks-llama-4-maverick",
        # ── removed (text-only): ──
        # "databricks-qwen3-next-80b-a3b-instruct",
        # "databricks-meta-llama-3-3-70b-instruct",
        # "databricks-meta-llama-3-1-8b-instruct",
        # "databricks-qwen35-122b-a10b",
        # "databricks-gpt-oss-120b",
        # "databricks-gpt-oss-20b",
    ],
}


def build_client(provider: str, model_id: str) -> LLMClient:
    if provider == "azure":
        return get_client("azure_openai", deployment=model_id, **AZURE_CONFIG)
    elif provider == "databricks":
        return get_client("databricks", endpoint_name=model_id)
    raise ValueError(f"Unknown provider: {provider}")


def _make_key(provider: str, model_id: str) -> str:
    # Avoid "databricks-databricks-..." duplication
    return model_id if model_id.startswith(provider) else f"{provider}-{model_id}"


# Each entry gets its OWN client bound to the correct deployment/endpoint
MODELS = {
    _make_key(provider, model_id): (build_client(provider, model_id), model_id)
    for provider, model_ids in PROVIDERS.items()
    for model_id in model_ids
}

In [0]:
MODELS

{'azure-gpt-5.5': (<__main__.AzureOpenAIClient at 0x7f84981ccf20>, 'gpt-5.5'),
 'azure-gpt-5.4': (<__main__.AzureOpenAIClient at 0x7f84981cecf0>, 'gpt-5.4'),
 'azure-gpt-5.2': (<__main__.AzureOpenAIClient at 0x7f8498ab5250>, 'gpt-5.2'),
 'azure-gpt-5.1': (<__main__.AzureOpenAIClient at 0x7f8498ab7830>, 'gpt-5.1'),
 'azure-gpt-5': (<__main__.AzureOpenAIClient at 0x7f84999fa420>, 'gpt-5'),
 'azure-gpt-4o': (<__main__.AzureOpenAIClient at 0x7f84c4163230>, 'gpt-4o'),
 'azure-gpt-4.1': (<__main__.AzureOpenAIClient at 0x7f84981cf650>, 'gpt-4.1'),
 'databricks-claude-opus-4-8': (<__main__.DatabricksServingClient at 0x7f8498b6d340>,
  'databricks-claude-opus-4-8'),
 'databricks-claude-opus-4-7': (<__main__.DatabricksServingClient at 0x7f8498ab4aa0>,
  'databricks-claude-opus-4-7'),
 'databricks-claude-opus-4-6': (<__main__.DatabricksServingClient at 0x7f8498b6e4b0>,
  'databricks-claude-opus-4-6'),
 'databricks-claude-opus-4-5': (<__main__.DatabricksServingClient at 0x7f8498b6d8e0>,
  'databri

In [0]:
from datetime import datetime
import time

N_REPEATS = 1
DELAY_SEC = 0.5

def run_rmet(include_definitions: bool = INCLUDE_DEFINITIONS):
    rows = []
    image_cache = {
        q["id"]: encode_image_to_base64(os.path.join(IMAGE_BASE, q["image"]))
        for q in QUESTIONS
    }

    for model_label, (client, model_name) in MODELS.items():
        for question in QUESTIONS:
            prompt = build_rmet_prompt(question, include_definitions)
            image_data_url = image_cache[question["id"]]

            for rep in range(N_REPEATS):
                try:
                    raw = client.query(prompt, image_data_url=image_data_url)
                    parsed = parse_rmet_response(raw, question["options"])
                except Exception as e:
                    parsed = {"choice": None, "raw": str(e), "parse_error": "API_ERROR"}

                is_correct = (
                    parsed["choice"] is not None
                    and parsed["choice"].lower() == question["correct"].lower()
                )

                rows.append({
                    "question_id": question["id"],
                    "image": question["image"],
                    "options": ", ".join(question["options"]),
                    "correct_answer": question["correct"],
                    "definitions_shown": include_definitions,   # ← key tracking column
                    "model_label": model_label,
                    "model": model_name,
                    "repeat": rep,
                    "choice": parsed["choice"],
                    "is_correct": is_correct,
                    "raw_response": parsed["raw"],
                    "parse_error": parsed["parse_error"],
                    "timestamp": datetime.now(),
                })
                time.sleep(DELAY_SEC)
    return rows


# Run both conditions and combine
results_no_defs   = run_rmet(include_definitions=False)
results_with_defs = run_rmet(include_definitions=True)
results = results_no_defs + results_with_defs

[Trace(request_id=tr-ff77c84ab8f441068c2ba08693d608f5), Trace(request_id=tr-00b09abd3ea74c0ab9c63d15e4d2e2dd), Trace(request_id=tr-c7bdadcccfad43ca9be06b5a702303a8), Trace(request_id=tr-9d58f843e17f4ef5b7c977a4eb2d1a64), Trace(request_id=tr-071aefc5089647689cc9cbda939ee327), Trace(request_id=tr-68d396fd3b0446ecb5fe3d131e831569), Trace(request_id=tr-9b4a6131bf314a62b60de7430da8dba1), Trace(request_id=tr-a59bfcac58b44e88949a643b9cd7cbbd), Trace(request_id=tr-9a7b43b594cc45f18f56819e0e0490ec), Trace(request_id=tr-f7276eefd3bc4a97ac5a8870267a9086)]

In [0]:
from pyspark.sql.types import *

schema = StructType([
    StructField("question_id", StringType()),
    StructField("image", StringType()),
    StructField("options", StringType()),
    StructField("correct_answer", StringType()),
    StructField("definitions_shown", BooleanType()),
    StructField("model_label", StringType()),
    StructField("model", StringType()),
    StructField("repeat", IntegerType()),
    StructField("choice", StringType()),
    StructField("is_correct", BooleanType()),
    StructField("raw_response", StringType()),
    StructField("parse_error", StringType()),
    StructField("timestamp", StringType()),
])

df = spark.createDataFrame(results, schema)
display(df)

question_id,image,options,correct_answer,definitions_shown,model_label,model,repeat,choice,is_correct,raw_response,parse_error,timestamp
rmet-1,eyes_1.png,"playful, comforting, irritated, bored",playful,false,azure-gpt-5.5,gpt-5.5,0,playful,true,"{""choice"": ""playful""}",null,2026-06-14 20:45:05.388254
rmet-2,eyes_2.png,"terrified, upset, arrogant, annoyed",upset,false,azure-gpt-5.5,gpt-5.5,0,arrogant,false,"{""choice"": ""arrogant""}",null,2026-06-14 20:45:11.838150
rmet-3,eyes_3.png,"joking, flustered, desire, convinced",desire,false,azure-gpt-5.5,gpt-5.5,0,desire,true,"{""choice"": ""desire""}",null,2026-06-14 20:45:18.789338
rmet-4,eyes_4.png,"joking, insisting, amused, relaxed",insisting,false,azure-gpt-5.5,gpt-5.5,0,insisting,true,"{""choice"": ""insisting""}",null,2026-06-14 20:45:24.863872
rmet-5,eyes_5.png,"irritated, sarcastic, worried, friendly",worried,false,azure-gpt-5.5,gpt-5.5,0,worried,true,"{""choice"": ""worried""}",null,2026-06-14 20:45:29.245201
rmet-6,eyes_6.png,"aghast, fantasizing, impatient, alarmed",fantasizing,false,azure-gpt-5.5,gpt-5.5,0,fantasizing,true,"{""choice"":""fantasizing""}",null,2026-06-14 20:45:33.036873
rmet-7,eyes_7.png,"apologetic, friendly, uneasy, dispirited",uneasy,false,azure-gpt-5.5,gpt-5.5,0,uneasy,true,"{""choice"": ""uneasy""}",null,2026-06-14 20:45:37.681717
rmet-8,eyes_8.png,"despondent, relieved, shy, excited",despondent,false,azure-gpt-5.5,gpt-5.5,0,relieved,false,"{""choice"": ""relieved""}",null,2026-06-14 20:45:43.213493
rmet-9,eyes_9.png,"annoyed, hostile, horrified, preoccupied",preoccupied,false,azure-gpt-5.5,gpt-5.5,0,hostile,false,"{""choice"": ""hostile""}",null,2026-06-14 20:45:49.896354
rmet-10,eyes_10.png,"cautious, insisting, bored, aghast",cautious,false,azure-gpt-5.5,gpt-5.5,0,insisting,false,"{""choice"":""insisting""}",null,2026-06-14 20:45:55.523874
